# TN0 — dựng lại kết quả MobiVital và chứng minh pipeline tương đương

Chạy **một mạch trên Colab**, khoảng 60 phút. Mọi logic nằm trong `scripts/`,
notebook chỉ gọi.

| bước | script | việc |
|---|---|---|
| 1 | `7_fix_old_filenames.py` | vá 52 tên file lỗi thời trong bảng MobiVital |
| 2 | `8_tn0_mobivital.py` | TN0a · TN0b · TN0c bằng **code MobiVital, 0 dòng sửa** |
| 3 | `9_tn0_ours.py` | TN0.1 bằng `src/scoring.py` của mình |
| 4 | `10_tn0_compare.py` | đối chiếu, phải trùng **537/537** |

## Ba bậc của TN0

```
TN0a   cham bang MobiVital commit san      -> so voi bai bao 0.819
TN0b   checkpoint cua ho -> tu sinh bang   -> moc cho TN0.1
TN0c   train lai tu dau  -> tu sinh bang   -> tai lap cong thuc train
```

Mỗi bậc thêm đúng một việc do mình tự làm. Bậc nào lệch đầu tiên thì lỗi nằm ở
đúng cái vừa thêm.

## Vì sao cần TN0.1

Code MobiVital chỉ chạy được LSTM — `mobivital_gen.py` dòng 152 ghi cứng
`LSTMMultiStep(...)`. Muốn thử TCN phải viết bộ chọn kênh riêng, rồi chứng minh
nó cho ra đúng kết quả code gốc.

## Cần chạy trước

`notebooks/DATA_PREPARE.ipynb`, **trong cùng phiên Colab này**. TN0 cần cả CSV
thô (13 GB, code MobiVital đọc thẳng) lẫn `by_user/*.npz`, mà CSV thô không cất
lên Drive. Ô setup bên dưới tự chạy lại nếu thiếu.

## 0. Setup

In [ ]:
import os
import subprocess


def run(command):
    """Chạy một lệnh shell, in ra những gì nó in."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())


REPO = "/content/UWB_RADAR"

if not os.path.exists(REPO + "/.git"):
    os.chdir("/content")
    run("rm -rf " + REPO)
    run("git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git " + REPO)
    run("git clone -q https://github.com/nesl/mobivital-public.git "
        + REPO + "/external/mobivital")
    run("pip install -q einops")
else:
    os.chdir(REPO)
    run("git pull -q origin main")

os.chdir(REPO)
run("git rev-parse --short HEAD")

### Dữ liệu

TN0 cần cả hai thứ:

```
data/raw/A..L                            CSV tho, code MobiVital doc thang
data/processed/by_user/*.npz             src/scoring.py doc
data/processed/mobivital_original/*.npy  autoreg_training.py doc
```

Thiếu thì ô dưới chạy lại `DATA_PREPARE` — mất khoảng 30 phút.

In [ ]:
if os.path.exists("data/raw/A") and os.path.exists("data/processed/by_user/A.npz"):
    print("dữ liệu đã có, bỏ qua")
else:
    print("thiếu dữ liệu, chạy lại các bước chuẩn bị...")
    run("apt-get install -qq -y aria2")
    run("aria2c -x16 -s16 -k5M --summary-interval=0 --console-log-level=error "
        "-d /content -o tripod.zip "
        "https://zenodo.org/api/records/15022885/files/tripod.zip/content")
    run("mkdir -p data/raw && unzip -q -o /content/tripod.zip -d data/raw/")
    run("python scripts/1_organize_raw.py | tail -2")
    run("python scripts/2_make_npz.py | tail -2")
    run("python scripts/3_run_mobivital_prep.py 2>&1 | tail -2")

run("du -sh data/raw data/processed/*")

## 1. Vá 52 tên file lỗi thời

Bảng kết quả MobiVital commit sẵn ra đời **trước khi dataset đổi tên lên Zenodo**.
52 dòng ghi mốc tháng 10, bản hiện tại là tháng 12:

```
trong bang : 231003_userI_tripod_01_0.csv
tren dia   : 231203_userI_tripod_01_0.csv
                 ^^
```

`evaluate.py` mở file theo tên trong bảng, gặp 52 tên đó là chết. Script tạo lối
tắt mang tên cũ trỏ vào file thật — không sửa bảng, không sửa code họ.

Dựng **hai** thư mục vì 52 tên đó đều thuộc G H I J, mà `mobivital_gen.py` duyệt
bằng `os.listdir()` nên nhét chung sẽ đếm thành 589 buổi ghi thay vì 537.

In [ ]:
!python scripts/7_fix_old_filenames.py

## 2. TN0a · TN0b · TN0c — code MobiVital, 0 dòng sửa

Script dựng một thư mục tạm có đúng những cái tên mà code họ đòi
(`./dataset/mobivital/tripod/`, `./data_final/`, `checkpoints/`), bên trong toàn
lối tắt, rồi `cd` vào đó mà gọi.

Cuối script in ra `git status` của repo MobiVital — phải trống.

Bước TN0c train lại từ đầu, khoảng **20 phút trên GPU**.

In [ ]:
!python scripts/8_tn0_mobivital.py

## 3. TN0.1 — bộ chọn kênh của mình

Cùng checkpoint LSTM của MobiVital, nhưng chạy qua `src/scoring.py`.

Với mỗi buổi ghi: dựng 240 ứng viên (120 kênh × 2 phép), lọc bằng
`invert_detector`, cắt 52 cửa sổ mỗi ứng viên, cho model dự báo, chọn kênh có
tổng Pearson cao nhất. **Không nhìn nhịp thở thật** ở bước chọn.

In [ ]:
!python scripts/9_tn0_ours.py

## 4. Đối chiếu — cửa ải

Cùng checkpoint, cùng dữ liệu, cùng thuật toán, không có gì ngẫu nhiên → phải ra
y hệt. Script dừng hẳn nếu lệch.

Trùng 537/537 chứng minh cùng lúc ba điều:

| | vì sao suy ra được |
|---|---|
| dữ liệu `by_user/*.npz` đúng | `scoring.py` đọc `.npz` còn `mobivital_gen.py` đọc CSV |
| bộ chọn kênh đúng | 537/537 |
| hàm chấm điểm đúng | 537 điểm khớp tới chữ số 15 |

In [ ]:
!python scripts/10_tn0_compare.py

## 5. Cất kết quả

Nén `results/` và `runs/tn0/` lên Drive, giữ nguyên cấu trúc thư mục để bung ở
máy cá nhân vào đúng chỗ:

```bash
# o may
tar -xzf ~/Downloads/tn0.tar.gz -C /Users/udnb/Desktop/THESIS_GRADUATE/
```

In [ ]:
run("mkdir -p /content/drive/MyDrive/mobivital")
run("tar -czf /content/drive/MyDrive/mobivital/tn0.tar.gz "
    "results runs/tn0/work/checkpoints runs/tn0/work/inference")
run("ls -la /content/drive/MyDrive/mobivital/tn0.tar.gz")
print()
run("ls -la results/")

## Xong

`results/` giờ có đủ, đối xứng với pipeline gốc:

```
                 lua chon kenh      diem tung buoi ghi
MobiVital        TN0b.txt           scores_TN0b.csv
                 TN0c.txt           scores_TN0c.csv
minh             TN0_1.txt          scores_TN0_1.csv
```

Từ đây thay LSTM bằng TCN, mọi khâu còn lại giữ nguyên.